# Suivi batch annotation OpenAI

Notebook léger pour vérifier si un job Batch est terminé et inspecter les fichiers locaux.

**Prérequis** : `OPENAI_API_KEY` dans `text/.env` pour rafraîchir le statut via l'API.

Workflow CLI complet :
```bash
python scripts/run_annotation_batch.py submit --config configs/annotation_batch.yaml
python scripts/run_annotation_batch.py status --run-id <RUN_ID>
python scripts/run_annotation_batch.py download --run-id <RUN_ID>
python scripts/run_annotation_batch.py ingest --run-id <RUN_ID>
```


In [1]:
import os
import sys
from pathlib import Path


def _notebook_find_text_root(start: Path) -> Path:
    here = start.resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "safer_core" / "paths.py").is_file():
            return candidate
        nested = candidate / "text"
        if (nested / "safer_core" / "paths.py").is_file():
            return nested
    raise FileNotFoundError(
        "Racine text/ introuvable (safer_core/paths.py). "
        "Ouvrez Jupyter depuis le dossier text/ ou SAFER/."
    )


TEXT_ROOT = _notebook_find_text_root(Path.cwd())
if str(TEXT_ROOT) not in sys.path:
    sys.path.insert(0, str(TEXT_ROOT))
os.chdir(TEXT_ROOT)


In [2]:
# --- Paramètres ---
# Remplacez par le RUN_ID affiché après submit (dossier sous annotation/outputs/).
RUN_ID = "run_partial_btp"
REFRESH_FROM_API = True   # False = lecture locale batch_state.json uniquement

ANNOTATION_ROOT = TEXT_ROOT / "annotation"
OUTPUTS_DIR = ANNOTATION_ROOT / "outputs" / RUN_ID
BATCH_STATE_PATH = OUTPUTS_DIR / "batch_state.json"
BATCH_OUTPUT_PATH = OUTPUTS_DIR / "batch_output.jsonl"
BATCH_ERRORS_PATH = OUTPUTS_DIR / "batch_errors.jsonl"

print("RUN_ID =", RUN_ID)
print("OUTPUTS_DIR =", OUTPUTS_DIR)
if not OUTPUTS_DIR.is_dir():
    raise FileNotFoundError(
        f"Dossier run introuvable : {OUTPUTS_DIR}\n"
        "Vérifiez RUN_ID (liste : annotation/outputs/)."
    )


RUN_ID = run_partial_btp
OUTPUTS_DIR = C:\Users\aho\Documents\analysis factor project\SAFER\text\annotation\outputs\run_partial_btp


In [3]:
# --- État local ---
import json
from pathlib import Path

state = {}
if BATCH_STATE_PATH.is_file():
    state = json.loads(BATCH_STATE_PATH.read_text(encoding="utf-8"))
    print("Statut (local) :", state.get("status"))
    print("Batch ID :", state.get("batch_id"))
    print("Soumis :", state.get("submitted_at"))
    print("Terminé :", state.get("completed_at"))
    counts = state.get("request_counts") or {}
    total = counts.get("total") or 0
    completed = counts.get("completed") or 0
    failed = counts.get("failed") or 0
    print(f"Progression : {completed}/{total} complétées, {failed} échecs")
    if total:
        pct = 100.0 * completed / total
        print(f"Avancement : {pct:.1f}%")
    state
else:
    print(f"batch_state.json introuvable : {BATCH_STATE_PATH}")
    if BATCH_OUTPUT_PATH.is_file():
        print("→ batch_output.jsonl est déjà présent : vous pouvez lancer l'ingest directement.")
    else:
        print("→ Exécutez submit/status via la CLI, ou corrigez RUN_ID.")


Statut (local) : in_progress
Batch ID : None
Soumis : None
Terminé : None
Progression : 34548/40000 complétées, 1 échecs
Avancement : 86.4%


In [4]:
# --- Rafraîchir depuis l'API OpenAI ---
from annotation.batch_config import load_batch_config_from_run
from annotation.batch_client import list_batch_chunks, refresh_batch_state

cfg = load_batch_config_from_run(RUN_ID, annotation_root=ANNOTATION_ROOT)

if REFRESH_FROM_API:
    state = refresh_batch_state(cfg)
    chunks = list_batch_chunks(state)
    counts = state.get("request_counts") or {}
    print("Statut global (API) :", state.get("status"))
    print(
        f"Progression API : {counts.get('completed', 0)}/{counts.get('total', 0)} "
        f"(failed={counts.get('failed', 0)})"
    )
    for chunk in chunks:
        ccounts = chunk.get("request_counts") or {}
        print(
            f"  - {chunk.get('batch_id')} : {chunk.get('status')} "
            f"({ccounts.get('completed', 0)}/{ccounts.get('total', 0)})"
        )
    if any(chunk.get("status") == "completed" for chunk in chunks) and not BATCH_OUTPUT_PATH.is_file():
        print("→ Au moins un chunk terminé côté OpenAI — lancez download + ingest.")
    state
else:
    print("REFRESH_FROM_API=False — statut local uniquement")


Statut global (API) : in_progress
Progression API : 34677/40000 (failed=1)
  - batch_6a67453f727881908094187530f289cd : completed (10000/10000)
  - batch_6a674fdabc348190a551a4b759cad9d8 : completed (10000/10000)
  - batch_6a6757a026308190b594400ad6e5da5c : completed (9999/10000)
  - batch_6a675bdbd84081909e7db8cee8aca34d : in_progress (4678/10000)


In [5]:
# --- Fichiers locaux ---
from pathlib import Path

def count_jsonl_lines(path: Path) -> int:
    if not path.is_file():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())

files = {
    "batch_state.json": BATCH_STATE_PATH.is_file(),
    "batch_output.jsonl": BATCH_OUTPUT_PATH.is_file(),
    "batch_errors.jsonl": BATCH_ERRORS_PATH.is_file(),
}
for name, exists in files.items():
    print(f"{name}: {'oui' if exists else 'non'}")

if BATCH_OUTPUT_PATH.is_file():
    print("Lignes batch_output :", count_jsonl_lines(BATCH_OUTPUT_PATH))
if BATCH_ERRORS_PATH.is_file():
    print("Lignes batch_errors :", count_jsonl_lines(BATCH_ERRORS_PATH))


batch_state.json: oui
batch_output.jsonl: oui
batch_errors.jsonl: oui
Lignes batch_output : 29999
Lignes batch_errors : 1


In [32]:
# --- Download + ingest (autonome : ne dépend pas des cellules précédentes) ---
from annotation.batch_config import load_batch_config_from_run
from annotation.batch_client import download_batch_results, list_batch_chunks, refresh_batch_state
from annotation.batch_runner import ingest_batch_results

cfg = load_batch_config_from_run(RUN_ID, annotation_root=ANNOTATION_ROOT)

if not BATCH_OUTPUT_PATH.is_file():
    state = refresh_batch_state(cfg)
    completed = [
        chunk for chunk in list_batch_chunks(state)
        if chunk.get("status") == "completed"
    ]
    if not completed:
        raise RuntimeError(
            f"Aucun chunk terminé (status global={state.get('status')!r}). "
            "Attendez la fin du batch OpenAI."
        )
    dl = download_batch_results(cfg, partial=True)
    print("Téléchargé :", dl)
else:
    print("batch_output.jsonl déjà présent — download ignoré.")

df_final, meta = ingest_batch_results(cfg)
print(meta)
df_final.head()


batch_output.jsonl déjà présent — download ignoré.


KeyboardInterrupt: 

In [ ]:
df_final.shape